In [ ]:
%cd ..

# Student Health Risk — Stacked HGBC / CatB / XGB / LGBM

**Goal**: Close the exploitable gap from the LightGBM-only baseline (~0.90 LB)
with complementary GBDT inductive biases + BA-optimised probability blending.

**Metric**: Balanced Accuracy (mean recall per class).

Inspired by
[kospintr/health-stacked-hgbc-catb-xgb-lgbm-baseline](https://www.kaggle.com/code/kospintr/health-stacked-hgbc-catb-xgb-lgbm-baseline).

| Layer | Choice |
|-------|--------|
| Base models | HistGradientBoosting, CatBoost, XGBoost, LightGBM |
| CV | Stratified 5-fold |
| Meta | `ModelWeightMeta` — Dirichlet simplex search on OOF probs |
| Preprocess | median/mode → label encode → interactions (`HealthPreprocessor`) |

> **Kaggle**: paste `scripts/kernels/stack.py` into a notebook cell (vendored,
> no `student_health` package needed). Locally this notebook imports from `src/`.

## 1. Setup

In [ ]:
from __future__ import annotations

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    ConfusionMatrixDisplay,
    confusion_matrix,
)
from sklearn.model_selection import StratifiedKFold
from tqdm.auto import tqdm

from student_health.data import load_data
from student_health.features import CAT_COLS, NUM_COLS, TARGET_COL, HealthPreprocessor
from student_health.models import (
    TARGET_LABELS,
    StackingEnsemble,
    build_meta_model,
    default_stack_base_models,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")

_DATA_CANDIDATES = [Path("data/raw"), Path("../data/raw")]
DATA_DIR = next((p for p in _DATA_CANDIDATES if (p / "train.csv").exists()), _DATA_CANDIDATES[0])
RANDOM_STATE = 42
N_FOLDS = 5
N_ESTIMATORS = 400  # drop to ~100 for a quick smoke run

print(f"✓ DATA_DIR = {DATA_DIR.resolve()}")
print(f"✓ N_ESTIMATORS = {N_ESTIMATORS}, N_FOLDS = {N_FOLDS}")

## 2. Load data

In [ ]:
train, test = load_data(DATA_DIR)
test_ids = test["id"].copy()

print(f"Train: {train.shape}  |  Test: {test.shape}")
print("Target distribution:")
print(train[TARGET_COL].value_counts(normalize=True).map("{:.1%}".format))
print()
print(f"Numeric: {NUM_COLS}")
print(f"Categorical: {CAT_COLS}")

## 3. Preprocess

Leakage-safe `HealthPreprocessor`: fit medians / modes / label encoders on
train only, then transform test. Adds the same interaction features as the
LightGBM baseline.

In [ ]:
prep = HealthPreprocessor()
X, feature_cols = prep.get_feature_matrix(train, fit=True)
X_test, _ = prep.get_feature_matrix(test, fit=False)
# Align columns
X_test = X_test.reindex(columns=feature_cols, fill_value=0)
y = train[TARGET_COL]

print(f"Features ({len(feature_cols)}): {feature_cols}")
print(f"X: {X.shape}  |  X_test: {X_test.shape}")

## 4. Build stack

Four complementary GBDT families with class weighting for the minority
`fit` / `unhealthy` recalls. Meta-learner searches simplex weights over the
four OOF probability tensors to maximise balanced accuracy (the WEIGHTING
path from the kospintr notebook).

In [ ]:
base_models = default_stack_base_models(
    random_state=RANDOM_STATE,
    n_estimators=N_ESTIMATORS,
)
meta = build_meta_model("model_weight", n_trials=1000, random_state=RANDOM_STATE)
cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

print("Base models:")
for name, est in base_models:
    print(f"  - {name}: {type(est).__name__}")
print(f"Meta: {type(meta).__name__}")

## 5. Train with stratified CV

In [ ]:
ensemble = StackingEnsemble(base_models=base_models, meta_model=meta)

# tqdm progress is logged inside StackingEnsemble via fold BA lines
with tqdm(total=1, desc="stack fit", unit="run") as pbar:
    ensemble.fit(X, y, cv=cv, X_test=X_test)
    pbar.update(1)

print()
print("=" * 60)
print("Per-model OOF BA:")
for name, score in ensemble.per_model_oof_scores_.items():
    print(f"  {name:12s} {score:.4f}")
print(f"Fold blend mean ± std: "
      f"{np.mean(ensemble.valid_scores_):.4f} ± {np.std(ensemble.valid_scores_):.4f}")
print(f"Meta OOF BA:           {ensemble.overall_oof_score_:.4f}")
if hasattr(ensemble.meta_model_, "weights_"):
    w = dict(zip([n for n, _ in base_models], ensemble.meta_model_.weights_))
    print(f"Blend weights:         { {k: round(float(v), 3) for k, v in w.items()} }")
print("=" * 60)

## 6. OOF evaluation

In [ ]:
oof_preds = ensemble.meta_model_.predict(ensemble.oof_meta_)
oof_labels = ensemble.label_encoder_.inverse_transform(oof_preds)
y_true = y.to_numpy()

print(f"OOF Balanced Accuracy: {ensemble.overall_oof_score_:.4f}")
print()
print(classification_report(y_true, oof_labels, labels=TARGET_LABELS, zero_division=0))

cm = confusion_matrix(y_true, oof_labels, labels=TARGET_LABELS)
disp = ConfusionMatrixDisplay(cm, display_labels=TARGET_LABELS)
disp.plot(cmap="Blues", values_format="d")
plt.title("OOF Confusion Matrix — Stack")
plt.show()

recall = cm.diagonal() / cm.sum(axis=1).clip(min=1)
print("Per-class recall:")
for label, r in zip(TARGET_LABELS, recall):
    print(f"  {label:12s}: {r:.4f}")

### 6.1  Base-model comparison

In [ ]:
rows = [
    {"model": name, "oof_ba": score}
    for name, score in ensemble.per_model_oof_scores_.items()
]
rows.append({"model": "meta_blend", "oof_ba": ensemble.overall_oof_score_})
cmp = pd.DataFrame(rows).sort_values("oof_ba")

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.barh(cmp["model"], cmp["oof_ba"], color="C0", alpha=0.8)
ax.axvline(ensemble.overall_oof_score_, color="C3", ls="--", label="meta")
ax.set(xlabel="OOF Balanced Accuracy", title="Base models vs meta blend")
ax.set_xlim(0.85, max(cmp["oof_ba"].max() + 0.01, 0.95))
ax.legend()
fig.tight_layout()
plt.show()
cmp

## 7. Submission

Uses cached fold-averaged test probabilities from `fit(..., X_test=...)`
so we do not re-infer all fold models.

In [ ]:
pred_labels = ensemble.predict()  # uses cached test_meta_
submission = pd.DataFrame({"id": test_ids, "health_condition": pred_labels})

out = Path("data/submissions")
out.mkdir(parents=True, exist_ok=True)
path = out / "submission_stack.csv"
submission.to_csv(path, index=False)

print(f"Saved {path} ({len(submission):,} rows)")
print(submission["health_condition"].value_counts())
submission.head(10)

### 7.1  Persist ensemble

In [ ]:
model_path = Path("outputs/stack_ensemble.joblib")
model_path.parent.mkdir(parents=True, exist_ok=True)
ensemble.save(model_path)
print(f"✓ saved {model_path}")

## 8. Next steps

1. Swap meta to `build_meta_model("hgbc")` for true stacking (HGBC on OOF probs).
2. Try `weighted_average` (per-model × per-class L-BFGS-B) for minority recall.
3. Tune `n_estimators` / early-stopping per family with Optuna on OOF BA.
4. Ensemble this stack with TabPFN OOF probabilities (complementary bias).
5. Threshold / class-multiplier tuning on OOF (`tune_thresholds` pattern).

**Kaggle paste path**: `scripts/kernels/stack.py` → single cell → GPU optional
(CPU is fine; CatBoost/XGB/LGBM/HGBC all run on CPU here).

In [ ]:
print("✓ Stack notebook complete")
print(f"  OOF Balanced Accuracy: {ensemble.overall_oof_score_:.4f}")
print(f"  Submission: data/submissions/submission_stack.csv")
print(f"  Kaggle kernel: scripts/kernels/stack.py")